In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib, os, json, gc
import pandas as pd
import numpy as np

PROJECT_ROOT = "/content/drive/MyDrive/nextgen_nlp_final"
ARTIFACTS_DIR = f"{PROJECT_ROOT}/artifacts"
pathlib.Path(ARTIFACTS_DIR).mkdir(parents=True, exist_ok=True)

CLEAN_V1_PATH = f"{ARTIFACTS_DIR}/clean_news_v1.parquet"
print("Exists:", os.path.exists(CLEAN_V1_PATH), CLEAN_V1_PATH)

Mounted at /content/drive
Exists: True /content/drive/MyDrive/nextgen_nlp_final/artifacts/clean_news_v1.parquet


In [2]:
df = pd.read_parquet(CLEAN_V1_PATH)
print("Shape:", df.shape)
print(df.columns.tolist())
df.head(2)

Shape: (197985, 3)
['date', 'title_clean', 'text_clean']


,date,title_clean,text_clean
0,2025-06-23 00:00:00+00:00,"Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod..."
1,2024-07-01 00:00:00+00:00,This AI video of gymnastics might be the freak...,This AI video of gymnastics might be the freak...


In [3]:
DATE_COL = "date"
TITLE_COL = "title_clean"
TEXT_COL = "text_clean"

In [4]:
from IPython.display import display

pd.set_option("display.max_colwidth", 3000)

samples = df.sample(3, random_state=42)[[DATE_COL, TITLE_COL, TEXT_COL]]
display(samples)

,date,title_clean,text_clean
35032,2025-06-28 00:00:00+00:00,Highlights of Stanford HAI's 2025 Artificial Intelligence Index Report | MEXC News,"Highlights of Stanford HAI's 2025 Artificial Intelligence Index Report | MEXC News MEXC Exchange/Crypto News/Highlights of Stanford HAI's 2025 Artificial Intelligence Index Report/Highlights of Stanford HAI's 2025 Artificial Intelligence Index ReportPANews2025/04/14 16:25MORE$0.01909+7.79%HAI$0.011143+0.09%AI$0.1102+0.45%INDEX$1.076+3.96%Author: Stanford HAI (Stanford Artificial Intelligence Institute) Compiled by: Felix, PANews Stanford HAI recently released the 456-page ""Artificial Intelligence Index Report 2025"". Here are some key points about artificial intelligence trends: 1. AI is becoming much more powerful than imagined In the new benchmarks MMMU, GPQA, and SWE-bench, AI performance improved significantly: scores increased by 18.8%, 48.9%, and 67.3%, respectively. In addition to the benchmarks, AI systems made significant progress in generating high-quality videos, and in some cases, large language models (LLMs) even surpassed humans in timed programming tasks. Note: MMMU is a novel, carefully designed benchmark for multi-disciplinary multimodal understanding and reasoning at the university level, aiming to evaluate the expert-level multimodal understanding capabilities of underlying models on a wide range of tasks. GPQA is a challenging dataset consisting of 448 high-quality and difficult multiple-choice questions written by experts in different fields. Experts who hold or are pursuing a PhD in the corresponding field achieve only 65% accuracy, while highly skilled non-expert verifiers achieve only 34% accuracy despite spending an average of more than 30 minutes and having unlimited access to the Internet. SWE-bench is a benchmark for evaluating the performance of Large Language Models (LLMs) on real-world software questions collected from GitHub. 2. AI is more efficient, accessible, and affordable Smaller AI models with fewer parameters are becoming increasingly powerful: in just two years, the number of parameters has been reduced by about 100 times, while still scoring over 60% on the Massive Multi-Task Language Understanding (MMLU) test. The gap between open source and closed source models is also narrowing, with the performance gap falling from 8% to just 1.7% in some benchmarks. Furthermore, the cost of inference for systems reaching the level of GPT-3.5 dropped by more than 280 times from November 2022 to October 2024. At the hardware level, costs dropped by 30% per year, while energy efficiency improved by 40% per year. The threshold for advanced AI is rapidly decreasing. Not to mention the development of sparse models like DeepSeek, where only relevant parameters are activated to answer the user’s query under the Mixture of Experts (MoE) structure, making the whole thing more efficient. Indeed, as smaller but more powerful AI models continue to emerge, the requirements for AI model training have been reduced, and cost-effective distributed training is expected to become mainstream in the next decade. There are currently some..."
112444,2025-05-15 00:00:00+00:00,Global supply chains get sustainability playbook powered by AI and expert consensus | Business,"Global supply chains get sustainability playbook powered by AI and expert consensus | Business HOME NEWS RESEARCH LIVE DISCOURSE BLOG / OPINION SUBMIT PRESS RELEASE About Career Advertisement Team Partnership Knowledge Partnership Media Partnership Contact Us NEWS RESEARCH LIVE DISCOURSE BLOG / OPINION INTERVIEW SUBMIT PRESS RELEASE Agro-Forestry Art & Culture Technology Economy Education Energy Politics Law & Governance Health Science Social Sports Transport Urban Development WASH Advertisement Home Blog Technology Article Global supply chains get sustainability playbook powered by AI and expert consensus The study identifies a clear pattern across all four dimensions of sustainable supply chain performance. In t

### 清洗

In [5]:
import re

email_re = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
phone_re = re.compile(r"(\+?\d{1,2}[\s.-]?)?(\(?\d{3}\)?[\s.-]?)\d{3}[\s.-]?\d{4}")

# 轻量模板短句（不要太激进，避免误删正文）
boiler_snippets = [
    r"subscribe\s+to\s+our\s+newsletter",
    r"sign\s+up\s+for\s+our\s+newsletter",
    r"privacy\s+policy",
    r"terms\s+of\s+service",
    r"cookie(s)?\s+policy",
    r"all\s+rights\s+reserved",
    r"advertisement",
    r"read\s+more",
]
boiler_re = re.compile("|".join(boiler_snippets), re.IGNORECASE)

ws_re = re.compile(r"\s+")

def light_clean(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = email_re.sub(" ", s)
    s = phone_re.sub(" ", s)
    s = boiler_re.sub(" ", s)
    s = ws_re.sub(" ", s).strip()
    return s

分块处理（避免一次性 map 20万行造成峰值）：

In [6]:
CHUNK = 20000
cleaned = []

texts = df[TEXT_COL].tolist()
for i in range(0, len(texts), CHUNK):
    block = texts[i:i+CHUNK]
    cleaned.extend([light_clean(x) for x in block])
    if (i//CHUNK) % 5 == 0:
        gc.collect()

df["text_clean2"] = cleaned
df.drop(columns=[TEXT_COL], inplace=True)
df.rename(columns={"text_clean2": TEXT_COL}, inplace=True)

del texts, cleaned
gc.collect()

print("After light regex clean:", df.shape)

After light regex clean: (197985, 3)


In [14]:
CLEAN_V2_PATH = f"{ARTIFACTS_DIR}/clean_news_v2_light_regex.parquet"

df.to_parquet(CLEAN_V2_PATH, index=False)

print("Saved:", CLEAN_V2_PATH)

Saved: /content/drive/MyDrive/nextgen_nlp_final/artifacts/clean_news_v2_light_regex.parquet


## sentence transformers

✅ 更高效、更稳、更强的方案（不用标注）

我们用：

🎯 语义过滤（Sentence Embedding Similarity）

这是：

无监督

不需要标注

不需要人工

完全合法

非常强

而且用 GPU 可以加速

这比关键词更高级，也比 Logistic 更优雅。

🧠 思路

我们定义一个“理想相关性描述”的文本，比如：

"Articles discussing how artificial intelligence impacts industries, companies, workforce, productivity, adoption, regulation, automation, and business transformation."


然后：

用 sentence-transformers（比如 all-MiniLM 或 bge-large）

把这句话 embedding 成向量

把所有文章（title+text前1000字符）embedding

计算 cosine similarity

设一个阈值，比如 top 30% 或 similarity > 0.35

只保留相似度高的

In [7]:
!pip install -q sentence-transformers

In [8]:
from sentence_transformers import SentenceTransformer
import torch

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
print("Using device:", device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Using device: cpu


In [11]:
query = """
Articles discussing how artificial intelligence impacts industries,
companies, workforce, productivity, adoption, automation,
regulation, business transformation, and enterprise deployment.
"""
query_emb = model.encode([query], normalize_embeddings=True)

In [13]:
import numpy as np

def embed_in_chunks(texts, batch_size=256):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        emb = model.encode(batch, normalize_embeddings=True)
        embeddings.append(emb)
    return np.vstack(embeddings)

combo_text = (df[TITLE_COL] + " " + df[TEXT_COL].str[:1000]).fillna("").tolist()

article_emb = embed_in_chunks(combo_text, batch_size=256)

KeyboardInterrupt: 

In [ ]:
similarities = article_emb @ query_emb.T
similarities = similarities.flatten()

df["semantic_score"] = similarities

In [ ]:
threshold = np.percentile(similarities, 70)
df_relevant = df[df["semantic_score"] >= threshold].copy()

print("Before:", df.shape)
print("After semantic filtering:", df_relevant.shape)
print("Keep rate:", len(df_relevant)/len(df))

In [ ]:
RELEVANT_PATH = f"{ARTIFACTS_DIR}/news_relevant_v1.parquet"
df_relevant[[DATE_COL, TITLE_COL, TEXT_COL]].to_parquet(RELEVANT_PATH, index=False)
print("Saved:", RELEVANT_PATH)